In [1]:
from math import gcd

DENOMINATIONS = [100, 200, 500, 1000, 2000, 5000]


def atm_withdrawal(amount, limits):
    if amount < 0 or len(limits) != len(DENOMINATIONS) or any(x < 0 for x in limits):
        raise ValueError("amount должен быть >= 0, а limits — 6 неотрицательных чисел")
    if amount == 0:
        return [0] * len(DENOMINATIONS)
    if amount % 100:
        return -1



    best_count = float("inf")
    best_result = None
    current = [0] * len(DENOMINATIONS)
    common_divisors = []
    divisor = 0
    for denomination, limit in zip(DENOMINATIONS, limits):
        divisor = gcd(divisor, denomination if limit else 0)
        common_divisors.append(divisor)
    failed_states = {}

    def search(index, remaining, used):
        nonlocal best_count, best_result
        if remaining == 0:
            if used < best_count:
                best_count = used
                best_result = current.copy()
            return
        if index < 0 or used >= best_count:
            return
        if common_divisors[index] == 0 or remaining % common_divisors[index] != 0:
            return
        state = (index, remaining)
        if failed_states.get(state, float("inf")) <= used:
            return
        failed_states[state] = used

        denomination = DENOMINATIONS[index]


        lower_bound = (remaining + denomination - 1) // denomination
        if used + lower_bound >= best_count:
            return
        max_count = min(limits[index], remaining // denomination)


        possible_value = sum(
            DENOMINATIONS[i] * limits[i] for i in range(index + 1)
        )
        if remaining > max_count * denomination + possible_value:
            return

        for count in range(max_count, -1, -1):
            next_used = used + count
            if next_used >= best_count:
                continue
            next_remaining = remaining - count * denomination
            if index > 0 and common_divisors[index - 1] and next_remaining % common_divisors[index - 1] != 0:
                continue
            if index > 0 and common_divisors[index - 1] == 0 and next_remaining != 0:
                continue
            if index == 0 and next_remaining != 0:
                continue
            current[index] = count
            search(index - 1, next_remaining, next_used)
        current[index] = 0

    search(len(DENOMINATIONS) - 1, amount, 0)
    return best_result if best_result is not None else -1


def gift_card_two_sum(prices, target):
    seen = {}
    for index, price in enumerate(prices):
        needed = target - price
        if needed in seen:
            return [seen[needed], index]
        seen[price] = index
    raise ValueError("Подходящая пара не найдена")


if __name__ == "__main__":
    print("Банкомат:", atm_withdrawal(700, [10, 10, 10, 10, 10, 10]))
    print("Банкомат:", atm_withdrawal(300, [0, 0, 0, 0, 0, 0]))
    print("Банкомат:", atm_withdrawal(350, [10, 0, 10, 10, 10, 10]))
    print("Банкомат:", atm_withdrawal(0, [0, 0, 0, 0, 0, 0]))
    print("Подарочная карта:", gift_card_two_sum([2, 7, 11, 15], 9))
    print("Подарочная карта:", gift_card_two_sum([3, 3], 6))


Банкомат: [0, 1, 1, 0, 0, 0]
Банкомат: -1
Банкомат: -1
Банкомат: [0, 0, 0, 0, 0, 0]
Подарочная карта: [0, 1]
Подарочная карта: [0, 1]


- `gift_card_two_sum`: среднее время `O(n)`, память `O(n)`.
- `atm_withdrawal`: в худшем случае перебирает произведение возможных количеств купюр по шести номиналам, время `O(∏ᵢ(min(limits[i], amount // denominations[i]) + 1))`; глубина рекурсии `O(6) = O(1)` при фиксированном числе номиналов. На максимальных ограничениях перебор может быть долгим.